# 06 — Single-stage protocol and fallback performance hardening

This targeted notebook requires the completed audited-v4 00–04
full run and corrected Notebook 05 v1.1 full run. It preserves
both evidence layers unchanged.

Run first with `HARDENING_MODE = "pilot"`. After it reports PASS,
change only that value to `"full"`, restart the runtime, and run
all again. The notebook contains one executable cell, embeds its
new module, validates both input packages, and has no synthetic
data or answer fallback.


In [ ]:
# Run pilot first. After PASS, change only HARDENING_MODE to "full".
HARDENING_MODE = "pilot"  # allowed: "pilot", "full"
ENVIRONMENT_LABEL = "colab_cpu_primary"
BASE_CLOSURE_ENVIRONMENT_LABEL = "colab_cpu_primary"

import subprocess
import sys
import importlib.metadata

required_packages = {
    "duckdb": "1.3.2",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "osqp": "1.0.4",
    "sqlglot": "27.0.0",
    "pyarrow": "19.0.1",
}
mismatched_packages = []
for package, required_version in required_packages.items():
    try:
        observed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        observed_version = "missing"
    if observed_version != required_version:
        mismatched_packages.append(f"{package}=={required_version}")
if mismatched_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched_packages]
    )
else:
    print("Pinned dependencies already available; installation skipped.")
for package, required_version in required_packages.items():
    observed_version = importlib.metadata.version(package)
    if observed_version != required_version:
        raise RuntimeError(
            f"Dependency version mismatch after installation: {package} "
            f"{observed_version} != {required_version}"
        )

from pathlib import Path
import hashlib
import importlib
import json
import os
import re
import shutil
import tempfile
import time
import zipfile

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root)).resolve()
active_path = PROJECT_ROOT / "active_run.json"
if not active_path.is_file():
    raise FileNotFoundError(
        "The audited full run is not active. Notebook 06 requires the completed "
        "audited 00-04 full run and Notebook 05 v1.1 full run."
    )
active = json.loads(active_path.read_text(encoding="utf-8"))
if active.get("run_mode") != "full":
    raise RuntimeError("Notebook 06 requires the audited full run.")
if active.get("suite_version") != "2026-08-01-audited-v4":
    raise RuntimeError("The active input is not audited-v4.")
ACTIVE_RUN_DIR = (PROJECT_ROOT / active["run_directory"]).resolve()
if PROJECT_ROOT not in ACTIVE_RUN_DIR.parents:
    raise RuntimeError("The active run path escaped the project root.")

def safe_extract_single_root(archive_path, expected_root, prefix):
    extraction_parent = Path(tempfile.mkdtemp(prefix=prefix))
    expected_prefix = expected_root + "/"
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        names = [info.filename for info in infos]
        if len(names) != len(set(names)):
            raise RuntimeError(f"Duplicate ZIP member in {archive_path.name}")
        if len(infos) > 10000:
            raise RuntimeError(f"Unexpected ZIP member count in {archive_path.name}")
        if sum(info.file_size for info in infos) > 2 * 1024**3:
            raise RuntimeError(f"ZIP exceeds the 2 GiB safety limit: {archive_path.name}")
        for info in infos:
            member = Path(info.filename)
            mode = (info.external_attr >> 16) & 0o170000
            if (
                info.flag_bits & 0x1
                or info.filename.startswith(("/", "\\"))
                or "\\" in info.filename
                or member.is_absolute()
                or ".." in member.parts
                or not info.filename.startswith(expected_prefix)
                or mode == 0o120000
            ):
                raise RuntimeError(f"Unsafe ZIP member: {info.filename}")
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
        archive.extractall(extraction_parent)
    return (extraction_parent / expected_root).resolve()

# Resolve the immutable Notebook 04 package.
evidence_package_name = "arbiter_icde_evidence_full_audited_v4"
evidence_markers = (
    "run_config.json",
    "checksums.csv",
    "checksums.sha256",
    "04_evidence/all_stage_quality_checks.csv",
    "code/arbiter_core.py",
)

def is_evidence_package(path):
    return path.is_dir() and all((path / marker).is_file() for marker in evidence_markers)

nested_evidence_dir = ACTIVE_RUN_DIR / evidence_package_name
if is_evidence_package(nested_evidence_dir):
    EVIDENCE_DIR = nested_evidence_dir.resolve()
    EVIDENCE_SOURCE = "nested_notebook04_package"
elif is_evidence_package(ACTIVE_RUN_DIR):
    EVIDENCE_DIR = ACTIVE_RUN_DIR
    EVIDENCE_SOURCE = "direct_extracted_package"
else:
    evidence_zip = ACTIVE_RUN_DIR / f"{evidence_package_name}.zip"
    if not evidence_zip.is_file():
        raise FileNotFoundError(
            "Notebook 04's audited evidence package is missing. Do not create "
            "manifests manually; rerun audited Notebook 04 in full mode."
        )
    EVIDENCE_DIR = safe_extract_single_root(
        evidence_zip,
        evidence_package_name,
        "arbiter-hardening-audited-v4-",
    )
    if not is_evidence_package(EVIDENCE_DIR):
        raise RuntimeError("The extracted Notebook 04 package is incomplete.")
    EVIDENCE_SOURCE = "verified_notebook04_zip"

# Resolve the completed Notebook 05 v1.1 package. The environment label is
# part of its deterministic directory/archive name.
closure_label_slug = re.sub(
    r"[^a-z0-9]+", "-", BASE_CLOSURE_ENVIRONMENT_LABEL.lower()
).strip("-")
if not closure_label_slug:
    raise ValueError("BASE_CLOSURE_ENVIRONMENT_LABEL must contain letters or digits.")
closure_root_name = f"05_request_bound_closure_full_{closure_label_slug}"
closure_markers = (
    "closure_run_config.json",
    "checksums.csv",
    "checksums.sha256",
    "stage_quality_checks.csv",
    "code/arbiter_request_release.py",
    "code/arbiter_closure_experiments.py",
)

def is_closure_package(path):
    return path.is_dir() and all((path / marker).is_file() for marker in closure_markers)

direct_closure_dir = ACTIVE_RUN_DIR / closure_root_name
if is_closure_package(direct_closure_dir):
    CLOSURE_DIR = direct_closure_dir.resolve()
    CLOSURE_SOURCE = "direct_notebook05_output"
else:
    closure_zip = (
        ACTIVE_RUN_DIR
        / f"arbiter_icde_closure_full_{closure_label_slug}_v1_1.zip"
    )
    if not closure_zip.is_file():
        raise FileNotFoundError(
            "Completed Notebook 05 v1.1 full evidence is missing. Run corrected "
            "Notebook 05 in full mode first. Do not copy checksums manually."
        )
    CLOSURE_DIR = safe_extract_single_root(
        closure_zip,
        closure_root_name,
        "arbiter-hardening-closure-v1-1-",
    )
    if not is_closure_package(CLOSURE_DIR):
        raise RuntimeError("The extracted Notebook 05 package is incomplete.")
    CLOSURE_SOURCE = "verified_notebook05_zip"

CODE_DIR = EVIDENCE_DIR / "code"
CLOSURE_CODE_DIR = CLOSURE_DIR / "code"
if not CODE_DIR.is_dir() or not CLOSURE_CODE_DIR.is_dir():
    raise FileNotFoundError("Pinned input code directories are missing.")

mode_config = {
    "pilot": {
        "protocol_repetitions": 3,
        "protocol_warmups": 1,
        "fallback_repetitions": 3,
        "fallback_warmups": 1,
        "concurrency_trials": 3,
        "bootstrap_resamples": 500,
    },
    "full": {
        "protocol_repetitions": 30,
        "protocol_warmups": 5,
        "fallback_repetitions": 50,
        "fallback_warmups": 5,
        "concurrency_trials": 50,
        "bootstrap_resamples": 10000,
    },
}
if HARDENING_MODE not in mode_config:
    raise ValueError("HARDENING_MODE must be 'pilot' or 'full'.")
options = mode_config[HARDENING_MODE]

# Materialize the exact embedded Notebook 06 module before importing it.
local_code_dir = Path(tempfile.mkdtemp(prefix="arbiter-performance-hardening-code-"))
embedded_sources = {
    "arbiter_performance_hardening.py": '"""Targeted performance hardening for the ARBITER closure campaign.\n\nThis module is deliberately downstream of two immutable evidence layers:\n\n* audited-v4 Notebooks 00--04 provide the supported SQL assets, compiled\n  constraints, controlled prices, and system-level evidence;\n* request-bound closure v1.1 (Notebook 05) provides the unchanged signed-token\n  protocol and its correctness evidence.\n\nNotebook 06 corrects a benchmark contract that staged an answer explicitly and\nthen staged it again inside ``RequestBoundSnapshotStore.issue_token``.  The\nprimary path below calls ``issue_token`` exactly once and instruments its one\ninternal durable stage.  The signed token format and release implementation are\nnot modified.\n\nThe fallback experiment also avoids a Python loop over satisfied constraint\nrows.  It reuses the audited active cyclic kernel with a budget equal to the\nentire current violation set, retains the independent verifier after every\ntier, and retains OSQP followed by rejection as the fail-closed backstop.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport secrets\nimport threading\nimport time\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nimport arbiter_core as core\nfrom arbiter_request_release import (\n    BoundAnswerBuffer,\n    DurableReleaseLedger,\n    RequestBoundSnapshotStore,\n    RequestContext,\n    canonical_answer_bytes,\n    exact_sql_digest,\n)\n\n\nAUDITED_SUITE_VERSION = "2026-08-01-audited-v4"\nCLOSURE_SUITE_VERSION = "2026-08-01-request-bound-closure-v1.1"\nHARDENING_SUITE_VERSION = "2026-08-12-performance-hardening-v1.0"\n\nCLOSURE_CODE_HASHES = {\n    "arbiter_request_release.py": "b00dc16f4ab5ea9fc57d30fb141c83f31003bc8014e8285c1e1eabd812f66fc3",\n    "arbiter_closure_experiments.py": "bf63760ee2124e6976bcf02e17d14c00c083defa4bdedae70fedfdec46e7cfb4",\n}\n\n\n# These are supported audited-v4 assets.  The row bands are preregistered\n# experiment categories, not discovered after timing.\nANSWER_PROFILE_SPECS: tuple[dict[str, Any], ...] = (\n    {\n        "size_class": "scalar",\n        "dataset": "TPC-H SF1",\n        "asset_id": "asset_2567eb4af28fbdb4",\n        "min_rows": 1,\n        "max_rows": 1,\n        "expected_rows": 1,\n    },\n    {\n        "size_class": "small",\n        "dataset": "UCI Adult",\n        "asset_id": "asset_ac656b81b6c77c9e",\n        "min_rows": 8,\n        "max_rows": 255,\n        "expected_rows": 21,\n    },\n    {\n        "size_class": "medium",\n        "dataset": "UCI Adult",\n        "asset_id": "asset_0009e06da018d508",\n        "min_rows": 1_024,\n        "max_rows": 9_999,\n        "expected_rows": 3_267,\n    },\n    {\n        "size_class": "large",\n        "dataset": "TPC-H SF1",\n        "asset_id": "asset_000f7a1dcff309ce",\n        "min_rows": 50_000,\n        "max_rows": 200_000,\n        "expected_rows": 93_750,\n    },\n)\n\n\ndef sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while True:\n            chunk = handle.read(chunk_bytes)\n            if not chunk:\n                break\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _boolean_series(series: pd.Series, *, name: str) -> pd.Series:\n    parsed = series.astype(str).str.lower().map({"true": True, "false": False})\n    if parsed.isna().any():\n        raise RuntimeError(f"{name} contains a non-boolean value")\n    return parsed.astype(bool)\n\n\ndef verify_closure_inputs(closure_dir: Path) -> pd.DataFrame:\n    """Fail closed on partial, stale, modified, or non-full Notebook 05 input."""\n\n    closure_dir = closure_dir.resolve()\n    required = [\n        closure_dir / "closure_run_config.json",\n        closure_dir / "checksums.csv",\n        closure_dir / "checksums.sha256",\n        closure_dir / "stage_quality_checks.csv",\n        closure_dir / "input_evidence_audit.csv",\n        closure_dir / "claim_audit.csv",\n        closure_dir / "code" / "code_manifest.json",\n    ]\n    for path in required:\n        if not path.is_file():\n            raise FileNotFoundError(f"required closure input is missing: {path}")\n\n    config = json.loads(\n        (closure_dir / "closure_run_config.json").read_text(encoding="utf-8")\n    )\n    if config.get("closure_suite_version") != CLOSURE_SUITE_VERSION:\n        raise RuntimeError("Notebook 06 requires request-bound closure v1.1")\n    if config.get("closure_mode") != "full":\n        raise RuntimeError("Notebook 06 requires a completed full Notebook 05 run")\n    if config.get("audited_input_suite_version") != AUDITED_SUITE_VERSION:\n        raise RuntimeError("Notebook 05 was not linked to audited-v4 input")\n    if config.get("audited_input_run_mode") != "full":\n        raise RuntimeError("Notebook 05 was not linked to a full audited-v4 run")\n\n    quality = pd.read_csv(closure_dir / "stage_quality_checks.csv")\n    if set(quality.columns) != {"check", "passed"}:\n        raise RuntimeError("Notebook 05 quality-check schema changed")\n    quality_passed = _boolean_series(quality["passed"], name="Notebook 05 checks")\n    if len(quality) != 25 or not quality_passed.all():\n        raise RuntimeError("the 25 Notebook 05 quality checks do not all pass")\n\n    prior_input_audit = pd.read_csv(closure_dir / "input_evidence_audit.csv")\n    if "passed" not in prior_input_audit.columns or not _boolean_series(\n        prior_input_audit["passed"], name="Notebook 05 input audit"\n    ).all():\n        raise RuntimeError("Notebook 05 did not validate all audited-v4 inputs")\n\n    claims = pd.read_csv(closure_dir / "claim_audit.csv")\n    request_claims = claims[\n        claims["claim"].astype(str).str.contains(\n            "Request-, buyer/session-", regex=False\n        )\n    ]\n    if len(request_claims) != 1 or request_claims.iloc[0]["status"] != "SUPPORTED":\n        raise RuntimeError("Notebook 05 request-binding closure is not supported")\n\n    manifest = pd.read_csv(closure_dir / "checksums.csv")\n    if set(manifest.columns) != {"path", "bytes", "sha256"}:\n        raise RuntimeError("Notebook 05 checksum-manifest schema changed")\n    if manifest.empty or manifest["path"].isna().any():\n        raise RuntimeError("Notebook 05 checksum manifest is empty or incomplete")\n    if manifest["path"].astype(str).duplicated().any():\n        raise RuntimeError("Notebook 05 checksum manifest has duplicate paths")\n    byte_counts = pd.to_numeric(manifest["bytes"], errors="coerce")\n    if byte_counts.isna().any() or (byte_counts < 0).any() or not np.all(\n        byte_counts.to_numpy(float) == np.floor(byte_counts.to_numpy(float))\n    ):\n        raise RuntimeError("Notebook 05 checksum manifest has invalid byte counts")\n    if not manifest["sha256"].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():\n        raise RuntimeError("Notebook 05 checksum manifest has invalid digests")\n\n    expected_sha_text = "".join(\n        f"{row.sha256}  {row.path}\\n" for row in manifest.itertuples(index=False)\n    )\n    if (closure_dir / "checksums.sha256").read_text(\n        encoding="utf-8"\n    ) != expected_sha_text:\n        raise RuntimeError("Notebook 05 CSV and text checksum manifests disagree")\n\n    rows: list[dict[str, Any]] = []\n    for record in manifest.itertuples(index=False):\n        relative = Path(str(record.path))\n        if relative.is_absolute() or ".." in relative.parts:\n            raise RuntimeError(f"unsafe Notebook 05 checksum path: {relative}")\n        target = (closure_dir / relative).resolve()\n        if closure_dir not in target.parents:\n            raise RuntimeError(f"Notebook 05 checksum path escaped: {relative}")\n        exists = target.is_file()\n        observed_bytes = target.stat().st_size if exists else -1\n        observed_hash = sha256_file(target) if exists else ""\n        passed = bool(\n            exists\n            and observed_bytes == int(record.bytes)\n            and observed_hash == str(record.sha256)\n        )\n        rows.append(\n            {\n                "scope": "closure_evidence",\n                "path": relative.as_posix(),\n                "expected_bytes": int(record.bytes),\n                "observed_bytes": int(observed_bytes),\n                "expected_sha256": str(record.sha256),\n                "observed_sha256": observed_hash,\n                "passed": passed,\n            }\n        )\n\n    code_manifest = json.loads(\n        (closure_dir / "code" / "code_manifest.json").read_text(encoding="utf-8")\n    )\n    if code_manifest != CLOSURE_CODE_HASHES:\n        raise RuntimeError("Notebook 05 code manifest is not the audited v1.1 manifest")\n    for name, expected_hash in CLOSURE_CODE_HASHES.items():\n        target = closure_dir / "code" / name\n        observed_hash = sha256_file(target) if target.is_file() else ""\n        rows.append(\n            {\n                "scope": "closure_code",\n                "path": f"code/{name}",\n                "expected_bytes": None,\n                "observed_bytes": target.stat().st_size if target.is_file() else -1,\n                "expected_sha256": expected_hash,\n                "observed_sha256": observed_hash,\n                "passed": observed_hash == expected_hash,\n            }\n        )\n\n    audit = pd.DataFrame(rows)\n    if audit.empty or not audit["passed"].all():\n        failed = audit.loc[~audit["passed"], "path"].tolist()\n        raise RuntimeError(f"Notebook 05 input integrity failed: {failed[:5]}")\n    return audit\n\n\ndef verify_database_and_source_inputs(\n    connection: Any,\n    *,\n    project_root: Path,\n    dataset_metadata: pd.DataFrame,\n    tpch_scale_factor: float,\n) -> pd.DataFrame:\n    """Recheck source hashes and every materialized table cardinality.\n\n    The earlier evidence package does not contain a content hash of the DuckDB\n    file itself.  This check therefore revalidates the two observational source\n    archives by SHA-256 and all ten materialized table counts.  It detects stale,\n    partial, differently scaled, or grossly substituted databases; a malicious\n    same-cardinality logical rewrite remains outside the available evidence and\n    must not be claimed as cryptographic database attestation.\n    """\n\n    from arbiter_data import (  # imported lazily from the pinned audited code\n        NYC_TAXI_ROWS,\n        UCI_ADULT_ROWS,\n        _expected_tpch_counts,\n    )\n\n    required_metadata = {\n        "dataset",\n        "local_file",\n        "sha256",\n        "total_rows",\n    }\n    if not required_metadata.issubset(dataset_metadata.columns):\n        raise ValueError("dataset metadata is missing required fields")\n    rows: list[dict[str, Any]] = []\n    for dataset in ["NYC Taxi 2024-01", "UCI Adult"]:\n        metadata = dataset_metadata[dataset_metadata["dataset"].eq(dataset)]\n        if len(metadata) != 1:\n            raise RuntimeError(f"dataset metadata is not unique: {dataset}")\n        record = metadata.iloc[0]\n        expected_hash = str(record["sha256"])\n        if len(expected_hash) != 64:\n            raise RuntimeError(f"missing source SHA-256 for {dataset}")\n        source_path = (project_root / "inputs" / Path(str(record["local_file"])).name).resolve()\n        if project_root not in source_path.parents:\n            raise RuntimeError(f"source input escaped project root: {source_path}")\n        observed_hash = sha256_file(source_path) if source_path.is_file() else ""\n        rows.append(\n            {\n                "scope": "source_file",\n                "name": dataset,\n                "expected": expected_hash,\n                "observed": observed_hash,\n                "passed": observed_hash == expected_hash,\n            }\n        )\n\n    expected_counts = {\n        "nyc_taxi": int(NYC_TAXI_ROWS),\n        "uci_adult": int(UCI_ADULT_ROWS),\n        **{\n            str(table): int(count)\n            for table, count in _expected_tpch_counts(tpch_scale_factor).items()\n        },\n    }\n    present = {\n        str(row[0])\n        for row in connection.execute(\n            "SELECT table_name FROM information_schema.tables "\n            "WHERE table_schema = current_schema()"\n        ).fetchall()\n    }\n    for table, expected_count in expected_counts.items():\n        exists = table in present\n        observed_count = (\n            int(connection.execute(f\'SELECT COUNT(*) FROM "{table}"\').fetchone()[0])\n            if exists\n            else -1\n        )\n        rows.append(\n            {\n                "scope": "duckdb_table_cardinality",\n                "name": table,\n                "expected": str(expected_count),\n                "observed": str(observed_count),\n                "passed": exists and observed_count == expected_count,\n            }\n        )\n    audit = pd.DataFrame(rows)\n    if len(audit) != 12 or not audit["passed"].all():\n        failed = audit.loc[~audit["passed"], "name"].tolist()\n        raise RuntimeError(f"database/source integrity failed: {failed[:5]}")\n    return audit\n\n\nclass InstrumentedDurableReleaseLedger(DurableReleaseLedger):\n    """Measure the one stage call nested inside ``issue_token``.\n\n    The subclass does not alter the transaction, schema, PRAGMAs, payload, or\n    return value.  Thread-local counters prevent unrelated worker threads from\n    contaminating a timing record.\n    """\n\n    def __init__(self, path: str | Path) -> None:\n        self._stage_metrics = threading.local()\n        super().__init__(path)\n        self.reset_stage_metrics()\n\n    def reset_stage_metrics(self) -> None:\n        self._stage_metrics.calls = 0\n        self._stage_metrics.elapsed_ns = 0\n\n    @property\n    def stage_calls(self) -> int:\n        return int(getattr(self._stage_metrics, "calls", 0))\n\n    @property\n    def stage_elapsed_ns(self) -> int:\n        return int(getattr(self._stage_metrics, "elapsed_ns", 0))\n\n    def stage_buffer(self, request: RequestContext, buffer: BoundAnswerBuffer) -> None:\n        started = time.perf_counter_ns()\n        try:\n            super().stage_buffer(request, buffer)\n        finally:\n            elapsed = time.perf_counter_ns() - started\n            self._stage_metrics.calls = self.stage_calls + 1\n            self._stage_metrics.elapsed_ns = self.stage_elapsed_ns + int(elapsed)\n\n\ndef execute_actual_answer_profiles(\n    assets: pd.DataFrame,\n    connection: Any,\n) -> tuple[dict[str, dict[str, Any]], pd.DataFrame]:\n    """Execute four preregistered supported assets against the audited database."""\n\n    required_columns = {"dataset", "asset_id", "sql", "expected_supported"}\n    if not required_columns.issubset(assets.columns):\n        raise ValueError("SQL asset table is missing required columns")\n    profiles: dict[str, dict[str, Any]] = {}\n    rows: list[dict[str, Any]] = []\n    for spec in ANSWER_PROFILE_SPECS:\n        matches = assets[\n            assets["dataset"].astype(str).eq(spec["dataset"])\n            & assets["asset_id"].astype(str).eq(spec["asset_id"])\n        ]\n        if len(matches) != 1:\n            raise RuntimeError(\n                f"answer profile asset is not unique: {spec[\'dataset\']}/{spec[\'asset_id\']}"\n            )\n        asset = matches.iloc[0]\n        if str(asset["expected_supported"]).lower() != "true":\n            raise RuntimeError(f"answer profile is not supported: {spec[\'asset_id\']}")\n        sql = str(asset["sql"])\n        started = time.perf_counter_ns()\n        answer = connection.execute(sql).fetchall()\n        query_ms = (time.perf_counter_ns() - started) / 1e6\n        answer_rows = len(answer)\n        if not int(spec["min_rows"]) <= answer_rows <= int(spec["max_rows"]):\n            raise RuntimeError(\n                f"{spec[\'size_class\']} profile returned {answer_rows} rows; "\n                f"expected [{spec[\'min_rows\']}, {spec[\'max_rows\']}]"\n            )\n        if answer_rows != int(spec["expected_rows"]):\n            raise RuntimeError(\n                f"{spec[\'size_class\']} profile cardinality changed: "\n                f"{answer_rows} != {spec[\'expected_rows\']}"\n            )\n        payload_bytes = canonical_answer_bytes(answer)\n        record = {\n            **spec,\n            "sql": sql,\n            "query_digest": exact_sql_digest(sql),\n            "sql_sha256": hashlib.sha256(sql.encode("utf-8")).hexdigest(),\n            "answer": answer,\n            "answer_rows": answer_rows,\n            "payload_bytes": len(payload_bytes),\n            "query_ms": query_ms,\n            "source": "DuckDB execution over audited NYC/Adult/TPC-H database",\n        }\n        profiles[str(spec["size_class"])] = record\n        rows.append({key: value for key, value in record.items() if key not in {"answer", "sql"}})\n    if set(profiles) != {"scalar", "small", "medium", "large"}:\n        raise RuntimeError("the four answer-size profiles were not constructed")\n    return profiles, pd.DataFrame(rows)\n\n\ndef _balanced_profile_order(profile_names: Sequence[str], repetition: int) -> list[str]:\n    names = list(profile_names)\n    if not names:\n        return []\n    rotation = int(repetition) % len(names)\n    order = names[rotation:] + names[:rotation]\n    if (int(repetition) // len(names)) % 2:\n        order = list(reversed(order))\n    return order\n\n\ndef _stable_seed(*parts: Any) -> int:\n    digest = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).digest()\n    return int.from_bytes(digest[:8], "big", signed=False)\n\n\ndef _bootstrap_median_ci(\n    values: Sequence[float],\n    *,\n    resamples: int,\n    seed: int,\n    confidence: float = 0.95,\n) -> tuple[float, float]:\n    vector = np.asarray(values, dtype=np.float64)\n    if vector.ndim != 1 or vector.size == 0 or not np.isfinite(vector).all():\n        raise ValueError("bootstrap input must be a nonempty finite vector")\n    if int(resamples) < 100:\n        raise ValueError("bootstrap requires at least 100 resamples")\n    generator = np.random.default_rng(int(seed))\n    indices = generator.integers(0, vector.size, size=(int(resamples), vector.size))\n    medians = np.median(vector[indices], axis=1)\n    alpha = (1.0 - float(confidence)) / 2.0\n    return (\n        float(np.quantile(medians, alpha, method="linear")),\n        float(np.quantile(medians, 1.0 - alpha, method="linear")),\n    )\n\n\nPROTOCOL_METRICS = (\n    "buffer_digest_ms",\n    "durable_stage_once_ms",\n    "verification_and_token_ms",\n    "stage_and_issue_total_ms",\n    "durable_commit_ms",\n    "delivery_integrity_ms",\n    "control_to_commit_ms",\n    "complete_to_delivery_ms",\n)\n\n\ndef summarize_protocol_timings(\n    raw: pd.DataFrame,\n    *,\n    bootstrap_resamples: int,\n) -> pd.DataFrame:\n    group_columns = ["size_class", "dataset", "asset_id"]\n    rows: list[dict[str, Any]] = []\n    for keys, group in raw.groupby(group_columns, sort=False):\n        row: dict[str, Any] = dict(zip(group_columns, keys, strict=True))\n        row.update(\n            {\n                "timing_repetitions": int(len(group)),\n                "answer_rows": int(group["answer_rows"].iloc[0]),\n                "payload_bytes": int(group["payload_bytes"].iloc[0]),\n                "stage_calls_min": int(group["stage_calls"].min()),\n                "stage_calls_max": int(group["stage_calls"].max()),\n                "release_rate": float(group["released"].mean()),\n                "durable_rate": float(group["durable"].mean()),\n            }\n        )\n        for metric in PROTOCOL_METRICS:\n            values = group[metric].to_numpy(dtype=np.float64)\n            low, high = _bootstrap_median_ci(\n                values,\n                resamples=bootstrap_resamples,\n                seed=_stable_seed("protocol", *keys, metric),\n            )\n            prefix = metric.removesuffix("_ms")\n            row[f"{prefix}_p50_ms"] = float(np.median(values))\n            row[f"{prefix}_p95_observed_ms"] = float(\n                np.quantile(values, 0.95, method="higher")\n            )\n            row[f"{prefix}_median_ci95_low_ms"] = low\n            row[f"{prefix}_median_ci95_high_ms"] = high\n        stage_seconds = max(float(np.median(group["durable_stage_once_ms"])) / 1000.0, 1e-12)\n        delivery_seconds = max(\n            float(np.median(group["delivery_integrity_ms"])) / 1000.0, 1e-12\n        )\n        payload_mib = int(group["payload_bytes"].iloc[0]) / (1024.0**2)\n        row["stage_throughput_mib_s"] = payload_mib / stage_seconds\n        row["delivery_verify_throughput_mib_s"] = payload_mib / delivery_seconds\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef run_single_stage_protocol_benchmark(\n    bundles: Mapping[str, Any],\n    profiles: Mapping[str, Mapping[str, Any]],\n    *,\n    work_dir: Path,\n    tolerance: float,\n    repetitions: int,\n    warmups: int,\n    bootstrap_resamples: int,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Benchmark one-stage v1.1 issue/commit/delivery over actual answers.\n\n    Store and schema construction occur outside each timed interval.  Every\n    repetition uses a fresh SQLite ledger, preventing file-growth history from\n    becoming an answer-size confounder.  Warmups are executed but never emitted.\n    """\n\n    if int(repetitions) < 1 or int(warmups) < 0:\n        raise ValueError("invalid protocol timing repetition counts")\n    work_dir.mkdir(parents=True, exist_ok=True)\n    profile_names = [str(spec["size_class"]) for spec in ANSWER_PROFILE_SPECS]\n    if set(profile_names) != set(profiles):\n        raise ValueError("protocol benchmark did not receive the four profile answers")\n\n    rows: list[dict[str, Any]] = []\n\n    def one(profile_name: str, repetition: int, measured: bool, execution_order: int) -> None:\n        profile = profiles[profile_name]\n        dataset = str(profile["dataset"])\n        bundle = bundles[dataset]\n        asset_id = str(profile["asset_id"])\n        asset_index = bundle.snapshot.asset_ids.index(asset_id)\n        quoted_price = float(bundle.prices[asset_index])\n        phase = "measured" if measured else "warmup"\n        ledger_path = work_dir / f"{phase}-{repetition:04d}-{profile_name}.sqlite"\n        for suffix in ["", "-wal", "-shm"]:\n            Path(str(ledger_path) + suffix).unlink(missing_ok=True)\n        ledger = InstrumentedDurableReleaseLedger(ledger_path)\n        store = RequestBoundSnapshotStore(\n            bundle.snapshot,\n            ledger=ledger,\n            asset_query_digests=bundle.query_digests,\n            secret=secrets.token_bytes(32),\n            verifier_tolerance=tolerance,\n        )\n        request = RequestContext.from_values(\n            request_id=f"hardening-{phase}-{repetition:04d}-{profile_name}",\n            buyer_id="hardening-buyer",\n            session_id=f"hardening-{phase}-session",\n            asset_id=asset_id,\n            sql=str(profile["sql"]),\n            quoted_price=quoted_price,\n        )\n\n        started = time.perf_counter_ns()\n        buffer = BoundAnswerBuffer(request=request, payload=profile["answer"])\n        buffer_ms = (time.perf_counter_ns() - started) / 1e6\n\n        ledger.reset_stage_metrics()\n        started = time.perf_counter_ns()\n        token = store.issue_token(bundle.snapshot, bundle.prices, request, buffer)\n        issue_total_ms = (time.perf_counter_ns() - started) / 1e6\n        stage_calls = ledger.stage_calls\n        stage_ms = ledger.stage_elapsed_ns / 1e6\n        token_ms = issue_total_ms - stage_ms\n        if stage_calls != 1 or token_ms < 0.0:\n            raise RuntimeError(\n                f"single-stage issue contract failed: calls={stage_calls}, residual_ms={token_ms}"\n            )\n\n        started = time.perf_counter_ns()\n        commit = store.compare_and_commit(token, bundle.prices, request, buffer)\n        commit_ms = (time.perf_counter_ns() - started) / 1e6\n\n        started = time.perf_counter_ns()\n        delivered = ledger.released_payload(token_id=token.token_id, request=request)\n        delivery_ms = (time.perf_counter_ns() - started) / 1e6\n        delivery_matches = delivered == buffer.payload_bytes\n        if not commit.released or not commit.durable or not delivery_matches:\n            raise RuntimeError("single-stage protocol benchmark failed release integrity")\n\n        if measured:\n            rows.append(\n                {\n                    "size_class": profile_name,\n                    "dataset": dataset,\n                    "asset_id": asset_id,\n                    "repetition": int(repetition),\n                    "execution_order": int(execution_order),\n                    "answer_rows": int(profile["answer_rows"]),\n                    "payload_bytes": int(len(buffer.payload_bytes)),\n                    "stage_calls": int(stage_calls),\n                    "duplicate_stage_detected": bool(stage_calls != 1),\n                    "buffer_digest_ms": float(buffer_ms),\n                    "durable_stage_once_ms": float(stage_ms),\n                    "verification_and_token_ms": float(token_ms),\n                    "stage_and_issue_total_ms": float(issue_total_ms),\n                    "durable_commit_ms": float(commit_ms),\n                    "delivery_integrity_ms": float(delivery_ms),\n                    "control_to_commit_ms": float(buffer_ms + issue_total_ms + commit_ms),\n                    "complete_to_delivery_ms": float(\n                        buffer_ms + issue_total_ms + commit_ms + delivery_ms\n                    ),\n                    "released": bool(commit.released),\n                    "durable": bool(commit.durable),\n                    "delivery_matches_buffer": bool(delivery_matches),\n                    "price_digest": token.price_digest,\n                    "constraint_digest": token.constraint_digest,\n                    "answer_digest": token.answer_digest,\n                }\n            )\n        ledger.close_thread_connection()\n        for suffix in ["", "-wal", "-shm"]:\n            Path(str(ledger_path) + suffix).unlink(missing_ok=True)\n\n    for warmup in range(int(warmups)):\n        for order, name in enumerate(_balanced_profile_order(profile_names, warmup)):\n            one(name, -warmup - 1, False, order)\n    for repetition in range(int(repetitions)):\n        for order, name in enumerate(_balanced_profile_order(profile_names, repetition)):\n            one(name, repetition, True, order)\n\n    raw = pd.DataFrame(rows)\n    summary = summarize_protocol_timings(\n        raw, bootstrap_resamples=int(bootstrap_resamples)\n    )\n    return raw, summary\n\n\n@dataclass\nclass OptimizedTieredRepairResult:\n    prices: np.ndarray\n    verification: core.VerificationResult\n    rejected: bool\n    fallback_used: bool\n    fallback_stage: str\n    solver: str\n    sweeps: int\n    projected_rows: int\n    runtime_ms: float\n    fallback_ms: float\n    l1_adjustment: float\n    error: str\n\n\ndef run_optimized_tiered_fail_safe_repair(\n    snapshot: core.ConstraintSnapshot,\n    candidate: np.ndarray,\n    *,\n    tolerance: float,\n    max_sweeps: int,\n    active_row_budget: int,\n    step: float,\n    osqp_max_iter: int,\n    violated_only_fallback_sweeps: int = 20,\n) -> OptimizedTieredRepairResult:\n    """Active repair -> violated-only cyclic -> OSQP -> reject."""\n\n    original = np.asarray(candidate, dtype=np.float64)\n    active = core.run_enforcement_method(\n        snapshot,\n        original,\n        method="arbiter_active",\n        tolerance=tolerance,\n        max_sweeps=max_sweeps,\n        active_row_budget=active_row_budget,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    if active.verification.passed:\n        return OptimizedTieredRepairResult(\n            prices=active.prices,\n            verification=active.verification,\n            rejected=False,\n            fallback_used=False,\n            fallback_stage="none",\n            solver=active.solver,\n            sweeps=active.sweeps,\n            projected_rows=active.projected_rows,\n            runtime_ms=active.runtime_ms,\n            fallback_ms=0.0,\n            l1_adjustment=float(np.abs(active.prices - original).sum()),\n            error=active.error,\n        )\n\n    # The active kernel already iterates only rows violated at the beginning of\n    # each sweep.  Expanding its budget to every current violation produces a\n    # full-coverage fallback without the legacy loop over satisfied rows.\n    violated_only = core.run_enforcement_method(\n        snapshot,\n        active.prices,\n        method="arbiter_active",\n        tolerance=tolerance,\n        max_sweeps=violated_only_fallback_sweeps,\n        active_row_budget=snapshot.constraints,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    total_runtime = active.runtime_ms + violated_only.runtime_ms\n    total_sweeps = active.sweeps + violated_only.sweeps\n    total_projected = active.projected_rows + violated_only.projected_rows\n    if violated_only.verification.passed:\n        return OptimizedTieredRepairResult(\n            prices=violated_only.prices,\n            verification=violated_only.verification,\n            rejected=False,\n            fallback_used=True,\n            fallback_stage="violated_only_cyclic",\n            solver="active_cyclic+violated_only_cyclic",\n            sweeps=total_sweeps,\n            projected_rows=total_projected,\n            runtime_ms=total_runtime,\n            fallback_ms=violated_only.runtime_ms,\n            l1_adjustment=float(np.abs(violated_only.prices - original).sum()),\n            error="; ".join(\n                part for part in [active.error, violated_only.error] if part\n            ),\n        )\n\n    qp = core.run_enforcement_method(\n        snapshot,\n        original,\n        method="osqp_qp",\n        tolerance=tolerance,\n        max_sweeps=max_sweeps,\n        active_row_budget=active_row_budget,\n        step=step,\n        allow_fallback=False,\n        osqp_max_iter=osqp_max_iter,\n    )\n    final_verification = core.independent_verify(\n        snapshot, qp.prices, tolerance=tolerance\n    )\n    rejected = not final_verification.passed\n    return OptimizedTieredRepairResult(\n        prices=qp.prices,\n        verification=final_verification,\n        rejected=rejected,\n        fallback_used=True,\n        fallback_stage="osqp" if not rejected else "rejected_after_osqp",\n        solver="active_cyclic+violated_only_cyclic+osqp",\n        sweeps=total_sweeps + qp.sweeps,\n        projected_rows=total_projected + qp.projected_rows,\n        runtime_ms=total_runtime + qp.runtime_ms,\n        fallback_ms=violated_only.runtime_ms + qp.runtime_ms,\n        l1_adjustment=float(np.abs(qp.prices - original).sum()),\n        error="; ".join(\n            part for part in [active.error, violated_only.error, qp.error] if part\n        ),\n    )\n\n\ndef _prior_fallback_conditions(\n    cases: Sequence[dict[str, Any]],\n    prior_budget_raw: pd.DataFrame,\n) -> list[dict[str, Any]]:\n    required = {\n        "dataset",\n        "relation_family",\n        "case_id",\n        "active_row_budget",\n        "variant",\n        "fallback_used",\n        "fallback_stage",\n        "oracle_verified",\n    }\n    if not required.issubset(prior_budget_raw.columns):\n        raise ValueError("Notebook 05 budget evidence schema changed")\n    fallback = prior_budget_raw[\n        prior_budget_raw["variant"].astype(str).eq("fail_safe")\n        & _boolean_series(\n            prior_budget_raw["fallback_used"], name="Notebook 05 fallback flags"\n        )\n    ].copy()\n    if fallback.empty:\n        raise RuntimeError("Notebook 05 contains no exercised fallback condition")\n    if not _boolean_series(\n        fallback["oracle_verified"], name="Notebook 05 fallback verification"\n    ).all():\n        raise RuntimeError("Notebook 05 contains an unverified fallback release")\n    if set(fallback["fallback_stage"].astype(str)) != {"full_cyclic"}:\n        raise RuntimeError("Notebook 05 fallback stage is not the expected full cyclic path")\n    key_columns = [\n        "dataset",\n        "relation_family",\n        "case_id",\n        "active_row_budget",\n    ]\n    unique = fallback[key_columns].drop_duplicates().sort_values(key_columns)\n    case_map = {\n        (str(case["dataset"]), str(case["relation_family"]), str(case["case_id"])): case\n        for case in cases\n    }\n    conditions: list[dict[str, Any]] = []\n    for row in unique.itertuples(index=False):\n        key = (str(row.dataset), str(row.relation_family), str(row.case_id))\n        if key not in case_map:\n            raise RuntimeError(f"could not reconstruct Notebook 05 fallback case: {key}")\n        conditions.append(\n            {\n                "dataset": key[0],\n                "relation_family": key[1],\n                "case_id": key[2],\n                "active_row_budget": int(row.active_row_budget),\n                "case": case_map[key],\n            }\n        )\n    return conditions\n\n\nFALLBACK_METHODS = (\n    "legacy_full_scan_cyclic",\n    "optimized_violated_only_cyclic",\n)\n\n\ndef summarize_fallback_timings(\n    raw: pd.DataFrame,\n    *,\n    bootstrap_resamples: int,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    group_columns = [\n        "dataset",\n        "relation_family",\n        "case_id",\n        "active_row_budget",\n        "method",\n    ]\n    summary_rows: list[dict[str, Any]] = []\n    for keys, group in raw.groupby(group_columns, sort=True):\n        values = group["runtime_ms"].to_numpy(dtype=np.float64)\n        low, high = _bootstrap_median_ci(\n            values,\n            resamples=bootstrap_resamples,\n            seed=_stable_seed("fallback", *keys),\n        )\n        summary_rows.append(\n            {\n                **dict(zip(group_columns, keys, strict=True)),\n                "timing_repetitions": int(len(group)),\n                "runtime_p50_ms": float(np.median(values)),\n                "runtime_p95_observed_ms": float(\n                    np.quantile(values, 0.95, method="higher")\n                ),\n                "runtime_median_ci95_low_ms": low,\n                "runtime_median_ci95_high_ms": high,\n                "verified_rate": float(group["oracle_verified"].mean()),\n                "unsafe_release_rate": float(group["unsafe_release"].mean()),\n                "projected_rows_median": float(group["projected_rows"].median()),\n                "sweeps_median": float(group["sweeps"].median()),\n            }\n        )\n    summary = pd.DataFrame(summary_rows)\n\n    key_columns = [\n        "dataset",\n        "relation_family",\n        "case_id",\n        "active_row_budget",\n        "repetition",\n    ]\n    pivot = raw.pivot(index=key_columns, columns="method", values="runtime_ms").reset_index()\n    if not set(FALLBACK_METHODS).issubset(pivot.columns):\n        raise RuntimeError("paired fallback timing rows are incomplete")\n    pivot["paired_speedup_legacy_over_optimized"] = (\n        pivot["legacy_full_scan_cyclic"]\n        / pivot["optimized_violated_only_cyclic"]\n    )\n    paired_rows: list[dict[str, Any]] = []\n    condition_columns = key_columns[:-1]\n    for keys, group in pivot.groupby(condition_columns, sort=True):\n        values = group["paired_speedup_legacy_over_optimized"].to_numpy(float)\n        low, high = _bootstrap_median_ci(\n            values,\n            resamples=bootstrap_resamples,\n            seed=_stable_seed("fallback-speedup", *keys),\n        )\n        paired_rows.append(\n            {\n                **dict(zip(condition_columns, keys, strict=True)),\n                "paired_repetitions": int(len(group)),\n                "paired_speedup_p50": float(np.median(values)),\n                "paired_speedup_p05_observed": float(\n                    np.quantile(values, 0.05, method="lower")\n                ),\n                "paired_speedup_median_ci95_low": low,\n                "paired_speedup_median_ci95_high": high,\n            }\n        )\n    return summary, pd.DataFrame(paired_rows)\n\n\ndef run_fallback_kernel_comparison(\n    cases: Sequence[dict[str, Any]],\n    prior_budget_raw: pd.DataFrame,\n    *,\n    tolerance: float,\n    max_sweeps: int,\n    step: float,\n    osqp_max_iter: int,\n    repetitions: int,\n    warmups: int,\n    bootstrap_resamples: int,\n    fallback_sweeps: int = 20,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Paired legacy-vs-optimized timing from identical active intermediates."""\n\n    if int(repetitions) < 1 or int(warmups) < 0:\n        raise ValueError("invalid fallback timing repetition counts")\n    conditions = _prior_fallback_conditions(cases, prior_budget_raw)\n    rows: list[dict[str, Any]] = []\n\n    def run_method(\n        condition: Mapping[str, Any],\n        active_prices: np.ndarray,\n        method: str,\n        repetition: int,\n        execution_order: int,\n        measured: bool,\n    ) -> None:\n        snapshot = condition["case"]["_snapshot"]\n        if method == "legacy_full_scan_cyclic":\n            core_method = "full_cyclic"\n        elif method == "optimized_violated_only_cyclic":\n            core_method = "arbiter_active"\n        else:\n            raise ValueError(f"unknown fallback method: {method}")\n        started = time.perf_counter_ns()\n        result = core.run_enforcement_method(\n            snapshot,\n            active_prices,\n            method=core_method,\n            tolerance=tolerance,\n            max_sweeps=fallback_sweeps,\n            active_row_budget=snapshot.constraints,\n            step=step,\n            allow_fallback=False,\n            osqp_max_iter=osqp_max_iter,\n        )\n        runtime_ms = (time.perf_counter_ns() - started) / 1e6\n        oracle = core.independent_verify(snapshot, result.prices, tolerance=tolerance)\n        if not oracle.passed or result.rejected:\n            raise RuntimeError(\n                f"fallback kernel failed verification: {condition[\'case_id\']}/{method}"\n            )\n        if measured:\n            candidate = condition["case"]["_candidate"]\n            rows.append(\n                {\n                    "dataset": condition["dataset"],\n                    "relation_family": condition["relation_family"],\n                    "case_id": condition["case_id"],\n                    "active_row_budget": int(condition["active_row_budget"]),\n                    "method": method,\n                    "repetition": int(repetition),\n                    "execution_order": int(execution_order),\n                    "runtime_ms": float(runtime_ms),\n                    "kernel_reported_ms": float(result.runtime_ms),\n                    "sweeps": int(result.sweeps),\n                    "projected_rows": int(result.projected_rows),\n                    "oracle_verified": bool(oracle.passed),\n                    "violations_after": int(oracle.violations),\n                    "max_residual_after": float(oracle.max_residual),\n                    "unsafe_release": bool(not result.rejected and not oracle.passed),\n                    "l1_adjustment_from_candidate": float(\n                        np.abs(result.prices - candidate).sum()\n                    ),\n                    "price_digest": core.price_vector_digest(\n                        snapshot.asset_ids, result.prices\n                    ),\n                    "candidate_digest": condition["case"]["candidate_digest"],\n                    "constraint_digest": snapshot.constraint_digest,\n                }\n            )\n\n    for condition_index, condition in enumerate(conditions):\n        case = condition["case"]\n        snapshot = case["_snapshot"]\n        active = core.run_enforcement_method(\n            snapshot,\n            case["_candidate"],\n            method="arbiter_active",\n            tolerance=tolerance,\n            max_sweeps=max_sweeps,\n            active_row_budget=int(condition["active_row_budget"]),\n            step=step,\n            allow_fallback=False,\n            osqp_max_iter=osqp_max_iter,\n        )\n        if active.verification.passed:\n            raise RuntimeError(\n                f"Notebook 05 fallback condition no longer requires fallback: {case[\'case_id\']}"\n            )\n        for warmup in range(int(warmups)):\n            order = list(FALLBACK_METHODS)\n            if (condition_index + warmup) % 2:\n                order.reverse()\n            for execution_order, method in enumerate(order):\n                run_method(\n                    condition,\n                    active.prices,\n                    method,\n                    -warmup - 1,\n                    execution_order,\n                    False,\n                )\n        for repetition in range(int(repetitions)):\n            order = list(FALLBACK_METHODS)\n            if (condition_index + repetition) % 2:\n                order.reverse()\n            for execution_order, method in enumerate(order):\n                run_method(\n                    condition,\n                    active.prices,\n                    method,\n                    repetition,\n                    execution_order,\n                    True,\n                )\n\n    raw = pd.DataFrame(rows)\n    summary, paired = summarize_fallback_timings(\n        raw, bootstrap_resamples=int(bootstrap_resamples)\n    )\n    return raw, summary, paired\n\n\ndef run_optimized_fail_safe_validation(\n    cases: Sequence[dict[str, Any]],\n    *,\n    budgets: Sequence[int],\n    tolerance: float,\n    max_sweeps: int,\n    step: float,\n    osqp_max_iter: int,\n) -> pd.DataFrame:\n    """Run the optimized tier once over the complete dense case/budget grid."""\n\n    rows: list[dict[str, Any]] = []\n    for case in cases:\n        snapshot = case["_snapshot"]\n        candidate = case["_candidate"]\n        precheck = core.independent_verify(snapshot, candidate, tolerance=tolerance)\n        for budget in budgets:\n            result = run_optimized_tiered_fail_safe_repair(\n                snapshot,\n                candidate,\n                tolerance=tolerance,\n                max_sweeps=max_sweeps,\n                active_row_budget=int(budget),\n                step=step,\n                osqp_max_iter=osqp_max_iter,\n            )\n            oracle = core.independent_verify(\n                snapshot, result.prices, tolerance=tolerance\n            )\n            rows.append(\n                {\n                    "dataset": case["dataset"],\n                    "relation_family": case["relation_family"],\n                    "case_id": case["case_id"],\n                    "active_row_budget": int(budget),\n                    "violations_before": int(precheck.violations),\n                    "budget_binding": bool(precheck.violations > int(budget)),\n                    "fallback_used": bool(result.fallback_used),\n                    "fallback_stage": result.fallback_stage,\n                    "solver": result.solver,\n                    "rejected": bool(result.rejected),\n                    "oracle_verified": bool(oracle.passed),\n                    "violations_after": int(oracle.violations),\n                    "max_residual_after": float(oracle.max_residual),\n                    "unsafe_release": bool(not result.rejected and not oracle.passed),\n                    "runtime_ms": float(result.runtime_ms),\n                    "fallback_ms": float(result.fallback_ms),\n                    "sweeps": int(result.sweeps),\n                    "projected_rows": int(result.projected_rows),\n                    "l1_adjustment": float(result.l1_adjustment),\n                    "candidate_digest": case["candidate_digest"],\n                    "constraint_digest": snapshot.constraint_digest,\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef hardening_quality_checks(\n    *,\n    mode: str,\n    audited_input_audit: pd.DataFrame,\n    closure_input_audit: pd.DataFrame,\n    database_input_audit: pd.DataFrame,\n    profile_catalog: pd.DataFrame,\n    protocol_raw: pd.DataFrame,\n    protocol_summary: pd.DataFrame,\n    protocol_faults: pd.DataFrame,\n    concurrency: pd.DataFrame,\n    delivery: pd.DataFrame,\n    fallback_raw: pd.DataFrame,\n    fallback_summary: pd.DataFrame,\n    fallback_paired: pd.DataFrame,\n    optimized_grid: pd.DataFrame,\n) -> pd.DataFrame:\n    """Fail-closed evidence checks for Notebook 06."""\n\n    if mode not in {"pilot", "full"}:\n        raise ValueError("mode must be pilot or full")\n    expected_protocol_repetitions = 30 if mode == "full" else 3\n    expected_fallback_repetitions = 50 if mode == "full" else 3\n    expected_concurrency_rows = 300 if mode == "full" else 18\n    checks: list[dict[str, Any]] = []\n\n    def add(name: str, passed: Any) -> None:\n        checks.append({"check": name, "passed": bool(passed)})\n\n    add("all audited-v4 input checks pass", audited_input_audit["passed"].all())\n    add("all Notebook 05 input checks pass", closure_input_audit["passed"].all())\n    add(\n        "observational source hashes and all ten database table counts pass",\n        len(database_input_audit) == 12 and database_input_audit["passed"].all(),\n    )\n    add(\n        "four preregistered actual DuckDB answer profiles are present",\n        len(profile_catalog) == 4\n        and set(profile_catalog["size_class"]) == {"scalar", "small", "medium", "large"}\n        and set(profile_catalog["source"])\n        == {"DuckDB execution over audited NYC/Adult/TPC-H database"},\n    )\n    band_pass = True\n    spec_by_class = {str(item["size_class"]): item for item in ANSWER_PROFILE_SPECS}\n    for row in profile_catalog.itertuples(index=False):\n        spec = spec_by_class.get(str(row.size_class))\n        band_pass = bool(\n            band_pass\n            and spec is not None\n            and int(spec["min_rows"]) <= int(row.answer_rows) <= int(spec["max_rows"])\n            and int(row.answer_rows) == int(spec["expected_rows"])\n        )\n    add("every answer profile satisfies its preregistered row band", band_pass)\n    add(\n        "protocol raw rows have unique profile/repetition keys",\n        not protocol_raw.duplicated(["size_class", "repetition"]).any(),\n    )\n    protocol_profile_consistency = True\n    for catalog_row in profile_catalog.itertuples(index=False):\n        timed = protocol_raw[\n            protocol_raw["size_class"].astype(str).eq(str(catalog_row.size_class))\n        ]\n        protocol_profile_consistency = bool(\n            protocol_profile_consistency\n            and not timed.empty\n            and timed["answer_rows"].nunique() == 1\n            and int(timed["answer_rows"].iloc[0]) == int(catalog_row.answer_rows)\n            and timed["payload_bytes"].nunique() == 1\n            and int(timed["payload_bytes"].iloc[0]) == int(catalog_row.payload_bytes)\n        )\n    add(\n        "protocol raw answer rows and bytes match the executed profile catalog",\n        protocol_profile_consistency,\n    )\n    add(\n        "protocol timing uses the required repetitions per answer profile",\n        len(protocol_raw) == 4 * expected_protocol_repetitions\n        and protocol_raw.groupby("size_class")["repetition"].nunique().eq(\n            expected_protocol_repetitions\n        ).all(),\n    )\n    add(\n        "every measured issue performs exactly one durable stage",\n        protocol_raw["stage_calls"].eq(1).all()\n        and not protocol_raw["duplicate_stage_detected"].any(),\n    )\n    add(\n        "every measured protocol path releases durably with matching delivery",\n        protocol_raw["released"].all()\n        and protocol_raw["durable"].all()\n        and protocol_raw["delivery_matches_buffer"].all(),\n    )\n    add(\n        "protocol summary reconciles raw repetition counts",\n        len(protocol_summary) == 4\n        and protocol_summary["timing_repetitions"].eq(\n            expected_protocol_repetitions\n        ).all(),\n    )\n    add(\n        "all 63 request-binding fault cases pass",\n        len(protocol_faults) == 63 and protocol_faults["passed"].all(),\n    )\n    add(\n        "all cross-service concurrency regressions pass",\n        len(concurrency) == expected_concurrency_rows and concurrency["passed"].all(),\n    )\n    add(\n        "all delivery-authorization regressions pass",\n        not delivery.empty and delivery["passed"].all(),\n    )\n    fallback_conditions = fallback_raw[\n        ["dataset", "relation_family", "case_id", "active_row_budget"]\n    ].drop_duplicates()\n    add(\n        "the four Notebook 05 fallback conditions are reconstructed",\n        len(fallback_conditions) == 4,\n    )\n    add(\n        "fallback timing rows have complete paired repetitions",\n        len(fallback_raw) == 4 * 2 * expected_fallback_repetitions\n        and not fallback_raw.duplicated(\n            [\n                "dataset",\n                "relation_family",\n                "case_id",\n                "active_row_budget",\n                "method",\n                "repetition",\n            ]\n        ).any(),\n    )\n    add(\n        "both fallback kernels independently verify every timing result",\n        fallback_raw["oracle_verified"].all()\n        and not fallback_raw["unsafe_release"].any(),\n    )\n    add(\n        "fallback summary reconciles raw repetitions",\n        len(fallback_summary) == 8\n        and fallback_summary["timing_repetitions"].eq(\n            expected_fallback_repetitions\n        ).all(),\n    )\n    add(\n        "optimized fallback is faster in every paired condition",\n        len(fallback_paired) == 4\n        and fallback_paired["paired_speedup_p50"].gt(1.0).all()\n        and fallback_paired["paired_speedup_median_ci95_low"].gt(1.0).all(),\n    )\n    add(\n        "optimized fail-safe grid has no unverified release",\n        len(optimized_grid) == 18 * 6\n        and optimized_grid["oracle_verified"].all()\n        and not optimized_grid["unsafe_release"].any(),\n    )\n    add(\n        "optimized fail-safe retains OSQP-or-reject backstop semantics",\n        set(optimized_grid["fallback_stage"]).issubset(\n            {"none", "violated_only_cyclic", "osqp", "rejected_after_osqp"}\n        ),\n    )\n\n    numeric_frames = {\n        "profile_catalog": profile_catalog,\n        "protocol_raw": protocol_raw,\n        "protocol_summary": protocol_summary,\n        "fallback_raw": fallback_raw,\n        "fallback_summary": fallback_summary,\n        "fallback_paired": fallback_paired,\n        "optimized_grid": optimized_grid,\n    }\n    for name, frame in numeric_frames.items():\n        numeric = frame.select_dtypes(include=[np.number])\n        add(\n            f"{name} contains no NaN or infinite numeric evidence",\n            np.isfinite(numeric.to_numpy(dtype=float)).all(),\n        )\n    return pd.DataFrame(checks)\n\n\ndef build_claim_audit(\n    *,\n    mode: str,\n    quality: pd.DataFrame,\n) -> pd.DataFrame:\n    passed = bool(quality["passed"].all())\n    full = mode == "full"\n    supported = "SUPPORTED" if passed and full else "NOT_SUPPORTED"\n    return pd.DataFrame(\n        [\n            {\n                "claim": "The measured v1.1 protocol path performs exactly one durable answer stage before request-bound commit.",\n                "status": supported,\n                "required_wording": "Limit to the four preregistered answer profiles and the recorded Colab environment.",\n            },\n            {\n                "claim": "Buffering, durable staging, authorization, commit, and delivery-integrity costs are separately measured for scalar, small, medium, and large actual DuckDB answers.",\n                "status": supported,\n                "required_wording": "Report answer rows and serialized bytes; do not generalize beyond the four profiles.",\n            },\n            {\n                "claim": "Violated-only cyclic fallback is faster than legacy full-row scanning on all four exercised Notebook 05 fallback conditions while preserving independent verification.",\n                "status": supported,\n                "required_wording": "Describe paired kernel-only measurements; do not call them end-to-end speedups.",\n            },\n            {\n                "claim": "P95 latency generalizes to other machines or workload distributions.",\n                "status": "NOT_SUPPORTED",\n                "required_wording": "Treat observed P95 as descriptive for one controlled machine run; report n and the quantile rule.",\n            },\n            {\n                "claim": "Network delivery, crash recovery, payment settlement, or distributed atomicity is established.",\n                "status": "NOT_SUPPORTED",\n                "required_wording": "The claim ends at a local SQLite authorization/outbox transaction and integrity-checked retrieval.",\n            },\n            {\n                "claim": "General SQL or standard TPC-H benchmark-query coverage is established.",\n                "status": "NOT_SUPPORTED",\n                "required_wording": "Retain the audited supported-fragment boundary.",\n            },\n        ]\n    )\n',
}
expected_embedded_hashes = {
    "arbiter_performance_hardening.py": "e4460d788b473225e913e0289d9e8f4b125be5f2b6b61095d66885026e764983",
}
for name, module_source in embedded_sources.items():
    module_path = local_code_dir / name
    module_path.write_text(module_source, encoding="utf-8")
    observed = hashlib.sha256(module_path.read_bytes()).hexdigest()
    if observed != expected_embedded_hashes[name]:
        raise RuntimeError(f"Embedded module hash mismatch: {name}")

for path in [str(local_code_dir), str(CLOSURE_CODE_DIR), str(CODE_DIR)]:
    if path in sys.path:
        sys.path.remove(path)
sys.path.insert(0, str(local_code_dir))
sys.path.insert(1, str(CLOSURE_CODE_DIR))
sys.path.insert(2, str(CODE_DIR))
for module_name in [
    "arbiter_performance_hardening",
    "arbiter_request_release",
    "arbiter_closure_experiments",
    "arbiter_core",
    "arbiter_experiments",
    "arbiter_workload",
    "arbiter_data",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import duckdb
import numpy as np
import pandas as pd

from arbiter_closure_experiments import (
    build_dataset_bundles,
    build_dense_candidate_catalog,
    hardware_environment_manifest,
    run_protocol_fault_campaign,
    verify_audited_inputs,
)
from arbiter_performance_hardening import (
    CLOSURE_CODE_HASHES,
    HARDENING_SUITE_VERSION,
    build_claim_audit,
    execute_actual_answer_profiles,
    hardening_quality_checks,
    run_fallback_kernel_comparison,
    run_optimized_fail_safe_validation,
    run_single_stage_protocol_benchmark,
    sha256_file,
    verify_closure_inputs,
    verify_database_and_source_inputs,
)

print("Startup import self-check PASS")
print("Active audited run:", ACTIVE_RUN_DIR)
print("Notebook 04 evidence:", EVIDENCE_DIR)
print("Notebook 04 source:", EVIDENCE_SOURCE)
print("Notebook 05 evidence:", CLOSURE_DIR)
print("Notebook 05 source:", CLOSURE_SOURCE)
print("Hardening mode:", HARDENING_MODE)

audited_input_audit = verify_audited_inputs(
    EVIDENCE_DIR,
    CODE_DIR,
    active_run_dir=ACTIVE_RUN_DIR,
)
closure_input_audit = verify_closure_inputs(CLOSURE_DIR)
print(f"Audited-v4 input integrity PASS ({len(audited_input_audit)} checks)")
print(f"Notebook 05 input integrity PASS ({len(closure_input_audit)} checks)")

run_config = json.loads(
    (EVIDENCE_DIR / "run_config.json").read_text(encoding="utf-8")
)
tolerance = float(run_config["tolerance"])
assets = pd.read_csv(EVIDENCE_DIR / "00_workload" / "sql_assets.csv")
relations = pd.read_csv(
    EVIDENCE_DIR / "00_workload" / "compiled_relations.csv"
)
policy = pd.read_csv(
    EVIDENCE_DIR / "01_proposals_attacks" / "controlled_policy.csv"
)
bundles = build_dataset_bundles(
    assets, relations, policy, tolerance=tolerance
)

database_path = (
    PROJECT_ROOT
    / "state"
    / f"arbiter_sf{run_config['tpch_scale_factor']:g}.duckdb"
)
if not database_path.is_file():
    raise FileNotFoundError(
        "The audited DuckDB database is missing. Rerun audited Notebook 00. "
        "Notebook 06 has no synthetic-data or synthetic-answer fallback."
    )
connection = duckdb.connect(str(database_path), read_only=True)
dataset_metadata = pd.read_csv(
    EVIDENCE_DIR / "00_workload" / "dataset_metadata.csv"
)
database_input_audit = verify_database_and_source_inputs(
    connection,
    project_root=PROJECT_ROOT,
    dataset_metadata=dataset_metadata,
    tpch_scale_factor=float(run_config["tpch_scale_factor"]),
)
print(f"Database/source integrity PASS ({len(database_input_audit)} checks)")

fault_answer_provenance = []
def execute_fault_answer(dataset, sql):
    started = time.perf_counter_ns()
    answer = connection.execute(sql).fetchall()
    elapsed_ms = (time.perf_counter_ns() - started) / 1e6
    fault_answer_provenance.append(
        {
            "phase": "protocol_fault_regression",
            "dataset": dataset,
            "sql_sha256": hashlib.sha256(sql.encode("utf-8")).hexdigest(),
            "answer_rows": len(answer),
            "query_ms": elapsed_ms,
            "source": "DuckDB execution over audited NYC/Adult/TPC-H database",
        }
    )
    return answer

label_slug = re.sub(r"[^a-z0-9]+", "-", ENVIRONMENT_LABEL.lower()).strip("-")
if not label_slug:
    raise ValueError("ENVIRONMENT_LABEL must contain letters or digits.")
output_name = f"06_performance_hardening_{HARDENING_MODE}_{label_slug}"
output_dir = (ACTIVE_RUN_DIR / output_name).resolve()
if ACTIVE_RUN_DIR not in output_dir.parents:
    raise RuntimeError("Notebook 06 output path escaped the audited run directory.")
if output_dir.exists():
    shutil.rmtree(output_dir)
for relative in ["input", "protocol", "performance", "code"]:
    (output_dir / relative).mkdir(parents=True, exist_ok=True)

hardening_config = {
    "hardening_suite_version": HARDENING_SUITE_VERSION,
    "hardening_mode": HARDENING_MODE,
    "environment_label": ENVIRONMENT_LABEL,
    "base_closure_environment_label": BASE_CLOSURE_ENVIRONMENT_LABEL,
    "audited_input_suite_version": run_config["suite_version"],
    "closure_input_suite_version": "2026-08-01-request-bound-closure-v1.1",
    "verifier_tolerance": tolerance,
    "answer_size_classes": ["scalar", "small", "medium", "large"],
    "active_row_budgets": [1, 4, 8, 16, 32, 64],
    "dense_target_violation_counts": [8, 192],
    "fallback_sweeps": 20,
    "timing_clock": "time.perf_counter_ns",
    "p95_rule": "empirical observed quantile using numpy method='higher'",
    "median_ci": "percentile bootstrap 95% CI with deterministic SHA-256-derived seeds",
    "outlier_policy": "retain every measured repetition; discard declared warmups only",
    "protocol_scope": "fresh SQLite ledger per repetition; schema/store construction excluded",
    **options,
}
(output_dir / "hardening_run_config.json").write_text(
    json.dumps(hardening_config, indent=2, sort_keys=True),
    encoding="utf-8",
)

profiles, profile_catalog = execute_actual_answer_profiles(assets, connection)
print("Actual answer profiles PASS")
display(profile_catalog[["size_class", "dataset", "asset_id", "answer_rows", "payload_bytes", "query_ms"]])

with tempfile.TemporaryDirectory(prefix="arbiter-hardening-work-") as temporary_work:
    temporary_work = Path(temporary_work)
    protocol_faults, concurrency, delivery, durable_records = (
        run_protocol_fault_campaign(
            bundles,
            answer_executor=execute_fault_answer,
            work_dir=temporary_work / "fault-regression",
            tolerance=tolerance,
            concurrency_trials=options["concurrency_trials"],
        )
    )
    protocol_raw, protocol_summary = run_single_stage_protocol_benchmark(
        bundles,
        profiles,
        work_dir=temporary_work / "single-stage-performance",
        tolerance=tolerance,
        repetitions=options["protocol_repetitions"],
        warmups=options["protocol_warmups"],
        bootstrap_resamples=options["bootstrap_resamples"],
    )

dense_cases, dense_catalog = build_dense_candidate_catalog(
    bundles,
    tolerance=tolerance,
    target_violation_counts=(8, 192),
    budget_fraction=0.50,
)
prior_budget_raw = pd.read_csv(
    CLOSURE_DIR / "budget" / "active_budget_trials_raw.csv"
)
fallback_raw, fallback_summary, fallback_paired = (
    run_fallback_kernel_comparison(
        dense_cases,
        prior_budget_raw,
        tolerance=tolerance,
        max_sweeps=int(run_config["repair_sweeps"]),
        step=float(run_config["repair_step"]),
        osqp_max_iter=int(run_config["osqp_max_iter"]),
        repetitions=options["fallback_repetitions"],
        warmups=options["fallback_warmups"],
        bootstrap_resamples=options["bootstrap_resamples"],
        fallback_sweeps=20,
    )
)
optimized_grid = run_optimized_fail_safe_validation(
    dense_cases,
    budgets=[1, 4, 8, 16, 32, 64],
    tolerance=tolerance,
    max_sweeps=int(run_config["repair_sweeps"]),
    step=float(run_config["repair_step"]),
    osqp_max_iter=int(run_config["osqp_max_iter"]),
)

quality = hardening_quality_checks(
    mode=HARDENING_MODE,
    audited_input_audit=audited_input_audit,
    closure_input_audit=closure_input_audit,
    database_input_audit=database_input_audit,
    profile_catalog=profile_catalog,
    protocol_raw=protocol_raw,
    protocol_summary=protocol_summary,
    protocol_faults=protocol_faults,
    concurrency=concurrency,
    delivery=delivery,
    fallback_raw=fallback_raw,
    fallback_summary=fallback_summary,
    fallback_paired=fallback_paired,
    optimized_grid=optimized_grid,
)
claims = build_claim_audit(mode=HARDENING_MODE, quality=quality)

audited_input_audit.to_csv(
    output_dir / "input" / "audited_v4_input_audit.csv", index=False
)
closure_input_audit.to_csv(
    output_dir / "input" / "notebook05_input_audit.csv", index=False
)
database_input_audit.to_csv(
    output_dir / "input" / "database_source_input_audit.csv", index=False
)
dense_catalog.to_csv(
    output_dir / "input" / "dense_candidate_catalog.csv", index=False
)
profile_catalog.to_csv(
    output_dir / "performance" / "answer_profile_catalog.csv", index=False
)
protocol_raw.to_csv(
    output_dir / "performance" / "single_stage_protocol_raw.csv", index=False
)
protocol_summary.to_csv(
    output_dir / "performance" / "single_stage_protocol_summary.csv", index=False
)
fallback_raw.to_csv(
    output_dir / "performance" / "fallback_kernel_raw.csv", index=False
)
fallback_summary.to_csv(
    output_dir / "performance" / "fallback_kernel_summary.csv", index=False
)
fallback_paired.to_csv(
    output_dir / "performance" / "fallback_paired_speedup.csv", index=False
)
optimized_grid.to_csv(
    output_dir / "performance" / "optimized_fail_safe_validation.csv", index=False
)
protocol_faults.to_csv(
    output_dir / "protocol" / "request_binding_fault_regression.csv", index=False
)
concurrency.to_csv(
    output_dir / "protocol" / "cross_service_concurrency_regression.csv", index=False
)
delivery.to_csv(
    output_dir / "protocol" / "delivery_authorization_regression.csv", index=False
)
durable_records.to_csv(
    output_dir / "protocol" / "durable_release_records.csv", index=False
)
pd.DataFrame(fault_answer_provenance).to_csv(
    output_dir / "protocol" / "fault_answer_provenance.csv", index=False
)
quality.to_csv(output_dir / "stage_quality_checks.csv", index=False)
claims.to_csv(output_dir / "claim_audit.csv", index=False)

environment = hardware_environment_manifest(extra=hardening_config)
(output_dir / "environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True), encoding="utf-8"
)
for name, module_source in embedded_sources.items():
    (output_dir / "code" / name).write_text(module_source, encoding="utf-8")
code_manifest = {
    **expected_embedded_hashes,
    **{f"notebook05/{name}": digest for name, digest in CLOSURE_CODE_HASHES.items()},
}
(output_dir / "code" / "code_manifest.json").write_text(
    json.dumps(code_manifest, indent=2, sort_keys=True), encoding="utf-8"
)

readme = f"""# ARBITER performance hardening {HARDENING_MODE} run

This targeted Notebook 06 campaign preserves audited-v4 Notebooks 00-04 and
request-bound closure Notebook 05 v1.1 unchanged. The signed token protocol
remains byte-identical to v1.1. The primary performance path calls
issue_token once and instruments its single internal durable stage; it never
performs the duplicate staging transaction present in the earlier benchmark.

Four preregistered supported assets produce scalar, small, medium, and large
answers by actual execution over the audited DuckDB database. Query execution
occurs once per profile and is recorded separately; it is not included in the
protocol control-path latency.

Fallback timings are paired kernel-only measurements from identical active
intermediates. The optimized path reuses the audited active cyclic kernel with
a budget covering all current violations, then independently verifies, invokes
OSQP if necessary, and otherwise rejects.

Observed P95 uses an empirical, non-interpolated order statistic (NumPy
method='higher'). Median 95% intervals use deterministic percentile bootstrap.
All measured repetitions are retained; only declared warmups are excluded.

Claims remain limited to this local SQLite outbox prototype, four answer
profiles, four exercised fallback conditions, and the recorded environment.
Network delivery, crash recovery, payment settlement, distributed atomicity,
general SQL, standard TPC-H queries, and cross-machine generalization remain
unsupported.

Environment label: {ENVIRONMENT_LABEL}
Hardening mode: {HARDENING_MODE}
Notebook 04 source: {EVIDENCE_SOURCE}
Notebook 05 source: {CLOSURE_SOURCE}
"""
(output_dir / "README.md").write_text(readme, encoding="utf-8")

if not quality["passed"].all():
    display(quality)
    failed = quality.loc[~quality["passed"], "check"].tolist()
    raise RuntimeError(
        "Notebook 06 quality checks failed. Do not use these results: "
        + "; ".join(failed)
    )

checksum_rows = []
for path in sorted(output_dir.rglob("*")):
    if path.is_file() and path.name not in {"checksums.csv", "checksums.sha256"}:
        checksum_rows.append(
            {
                "path": path.relative_to(output_dir).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
checksum_frame = pd.DataFrame(checksum_rows)
checksum_frame.to_csv(output_dir / "checksums.csv", index=False)
(output_dir / "checksums.sha256").write_text(
    "".join(
        f"{row.sha256}  {row.path}\n"
        for row in checksum_frame.itertuples(index=False)
    ),
    encoding="utf-8",
)
for row in checksum_frame.itertuples(index=False):
    target = output_dir / row.path
    if target.stat().st_size != int(row.bytes) or sha256_file(target) != row.sha256:
        raise RuntimeError(f"Post-write checksum verification failed: {row.path}")

archive_path = (
    ACTIVE_RUN_DIR
    / f"arbiter_icde_performance_hardening_{HARDENING_MODE}_{label_slug}_v1_0.zip"
)
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9
) as archive:
    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                (Path(output_dir.name) / path.relative_to(output_dir)).as_posix(),
            )
with zipfile.ZipFile(archive_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP verification failed at {bad_member}")

connection.close()
display(quality)
display(
    protocol_summary[[
        "size_class",
        "answer_rows",
        "payload_bytes",
        "complete_to_delivery_p50_ms",
        "complete_to_delivery_p95_observed_ms",
    ]]
)
display(
    fallback_paired[[
        "dataset",
        "active_row_budget",
        "paired_speedup_p50",
        "paired_speedup_median_ci95_low",
        "paired_speedup_median_ci95_high",
    ]]
)
print("Performance hardening notebook PASS")
print("Return this ZIP for independent audit:")
print(archive_path)
